In [ ]:
def predict_box(model, processor, screenshot_path: Path, goal: str, som_map: Dict[str, str], dom_path: Path) -> Dict[str, Any]:
    prompt = build_prompt(goal, som_map, str(dom_path))
    image = Image.open(screenshot_path).convert("RGB")
    image.thumbnail((896, 896))

    image_token = _get_image_token(processor, model)
    text = f"{image_token}\n{prompt}"

    inputs = processor(images=image, text=text, return_tensors="pt", truncation=True, max_length=512, padding=False)
    if hasattr(model, "device"):
        inputs = {k: v.to(model.device) if hasattr(v, "to") else v for k, v in inputs.items()}

    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=20, do_sample=False)
    raw = processor.decode(output[0], skip_special_tokens=True)
    match = re.search(r"\b(\d+)\b", raw.split(prompt)[-1])
    return {
        "prompt": prompt,
        "raw": raw,
        "predicted_box": int(match.group(1)) if match else None,
    }
{
    "cells": [
        {
            "cell_type": "markdown",
            "id": "#VSC-830fab4d",
            "metadata": {
                "language": "markdown"
            },
            "source": [
                "# Playwright Timeout Debug Notebook",
                "",
                "This notebook reproduces and diagnoses the `Page.goto` timeout path in `run_evaluator.py`, while also letting you test the Gemma-4 E4B adapter directly with screenshots, DOM context, and SoM boxes.",
                "",
                "Suggested flow:",
                "1. Set your environment variables and paths.",
                "2. Reproduce the evaluator failure from inside the notebook.",
                "3. Capture browser artifacts and inspect screenshots.",
                "4. Load the model and run one prediction.",
                "5. Experiment with retry and timeout strategies.",
                "6. Run the lightweight local-server tests before trying Adobe again."
            ]
        },
        {
            "cell_type": "code",
            "id": "#VSC-67df7e23",
            "metadata": {
                "language": "python"
            },
            "source": [
                ""
            ]
        },
        {
            "cell_type": "code",
            "id": "#VSC-693b55d0",
            "metadata": {
                "language": "python"
            },
            "source": [
                ""
            ]
        },
        {
            "cell_type": "code",
            "id": "#VSC-726bc54a",
            "metadata": {
                "language": "python"
            },
            "source": [
                ""
            ]
        },
        {
            "cell_type": "code",
            "id": "#VSC-aa5cd1bb",
            "metadata": {
                "language": "python"
            },
            "source": [
                ""
            ]
        },
        {
            "cell_type": "code",
            "id": "#VSC-c286b25d",
            "metadata": {
                "language": "python"
            },
            "source": [
                ""
            ]
        }
    ]
}

# Playwright Timeout Debug Notebook

This notebook reproduces and diagnoses the `Page.goto` timeout path in `run_evaluator.py`, while also letting you test the Gemma-4 E4B adapter directly with screenshots, DOM context, and SoM boxes.

Suggested flow:
1. Set your environment variables and paths.
2. Reproduce the evaluator failure from inside the notebook.
3. Capture browser artifacts and inspect screenshots.
4. Load the model and run one prediction.
5. Experiment with retry and timeout strategies.
6. Run the lightweight local-server tests before trying Adobe again.

In [ ]:
def debug_vision_capability(model, processor, image_path: Path) -> Dict[str, Any]:
    info: Dict[str, Any] = {}

    cfg_obj = getattr(model, "config", None)
    info["model_class"] = model.__class__.__name__
    info["model_type"] = getattr(cfg_obj, "model_type", None) if cfg_obj is not None else None
    info["has_vision_config"] = bool(getattr(cfg_obj, "vision_config", None)) if cfg_obj is not None else False

    tokenizer = getattr(processor, "tokenizer", None)
    image_token_id = getattr(cfg_obj, "image_token_index", None) if cfg_obj is not None else None
    if image_token_id is None and tokenizer is not None:
        for candidate in ["<image>", "<|image|>", "<img>"]:
            try:
                tok_id = tokenizer.convert_tokens_to_ids(candidate)
            except Exception:
                tok_id = None
            if tok_id is not None and tok_id != getattr(tokenizer, "unk_token_id", None):
                image_token_id = tok_id
                break
    info["image_token_id"] = image_token_id

    info["processor_has_image_processor"] = hasattr(processor, "image_processor")
    info["processor_keys"] = sorted([k for k in dir(processor) if "image" in k.lower() or "template" in k.lower()])[:25]

    image = Image.open(image_path).convert("RGB")
    image.thumbnail((896, 896))

    prompt = "Describe this page in one short sentence."
    image_token = _get_image_token(processor, model)
    text = f"{image_token}\n{prompt}"
    info["image_token_text"] = image_token
    info["used_chat_template"] = False

    inputs = processor(images=image, text=text, return_tensors="pt", truncation=True, max_length=512, padding=False)
    info["input_keys"] = sorted(inputs.keys())
    input_ids = inputs.get("input_ids")
    info["input_ids_shape"] = tuple(input_ids.shape) if input_ids is not None else None

    image_token_count = None
    if input_ids is not None and image_token_id is not None:
        image_token_count = int((input_ids == image_token_id).sum().item())
    info["image_token_count"] = image_token_count

    if hasattr(model, "device"):
        inputs = {k: v.to(model.device) if hasattr(v, "to") else v for k, v in inputs.items()}

    try:
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=8, do_sample=False)
        decoded = processor.decode(out[0], skip_special_tokens=True)
        info["generate_ok"] = True
        info["sample_output"] = decoded[:300]
    except Exception as exc:
        info["generate_ok"] = False
        info["generate_error"] = str(exc)

    return info

# Example debug call (run this cell after loading model/processor):
# vision_debug = debug_vision_capability(model, processor, artifact_info["screenshot_path"])
# vision_debug